# **NLP Pipeline for Basic Text Classification**

In [57]:
pip install pandas numpy scikit-learn nltk joblib

In [58]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [59]:
data = {
    "text": [
        # POSITIVE
        "I love this product! It's amazing.",
        "Very good quality and fast delivery.",
        "I'm extremely happy with the purchase!",
        "Excellent service and great value for money.",

        # NEGATIVE
        "Worst experience ever. I hate it.",
        "Terrible service, not recommended.",
        "This is a waste of money.",
        "The product broke after one day. Very disappointed.",

        # NEUTRAL
        "The product arrived yesterday.",
        "It works as described.",
        "Delivery was on time.",
        "The packaging was okay."
    ],
    "label": [
        "positive", "positive", "positive", "positive",
        "negative", "negative", "negative", "negative",
        "neutral", "neutral", "neutral", "neutral"
    ]
}



df = pd.DataFrame(data)
print(df.head())


                                           text     label
0            I love this product! It's amazing.  positive
1          Very good quality and fast delivery.  positive
2        I'm extremely happy with the purchase!  positive
3  Excellent service and great value for money.  positive
4             Worst experience ever. I hate it.  negative


In [60]:
print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nLabel distribution:\n", df["label"].value_counts())


Dataset shape: (12, 2)

Missing values:
 text     0
label    0
dtype: int64

Label distribution:
 label
positive    4
negative    4
neutral     4
Name: count, dtype: int64


In [61]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)      # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)           # keep only letters
    text = re.sub(r"\s+", " ", text).strip()       # remove extra spaces
    return text


In [62]:
df["clean_text"] = df["text"].apply(clean_text)

print(df[["text", "clean_text"]].head())


                                           text  \
0            I love this product! It's amazing.   
1          Very good quality and fast delivery.   
2        I'm extremely happy with the purchase!   
3  Excellent service and great value for money.   
4             Worst experience ever. I hate it.   

                                    clean_text  
0              i love this product its amazing  
1          very good quality and fast delivery  
2         im extremely happy with the purchase  
3  excellent service and great value for money  
4              worst experience ever i hate it  


In [63]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Train size: 9
Test size: 3


In [64]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),  # unigrams + bigrams
        stop_words="english"
    )),
    ("classifier", LogisticRegression(max_iter=1000))
])


In [65]:
model.fit(X_train, y_train)
print("Model training complete!")


Model training complete!


In [66]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test.values)


Predictions: ['neutral' 'positive' 'positive']
Actual: ['positive' 'neutral' 'negative']


In [67]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.0


In [68]:
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))



Confusion Matrix:
 [[0 0 1]
 [0 0 1]
 [0 1 0]]


In [69]:
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))



Classification Report:

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       1.0
     neutral       0.00      0.00      0.00       1.0
    positive       0.00      0.00      0.00       1.0

    accuracy                           0.00       3.0
   macro avg       0.00      0.00      0.00       3.0
weighted avg       0.00      0.00      0.00       3.0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [70]:
joblib.dump(model, "text_classifier.pkl")
print("Model saved as text_classifier.pkl")


Model saved as text_classifier.pkl


In [71]:
loaded_model = joblib.load("text_classifier.pkl")

sample_text = ["This product is fantastic and worth the money!"]
sample_text_clean = [clean_text(t) for t in sample_text]

prediction = loaded_model.predict(sample_text_clean)
print("Prediction:", prediction[0])


Prediction: positive


In [72]:
sample_text = ["I hate You"]
sample_text_clean = [clean_text(t) for t in sample_text]

prediction = loaded_model.predict(sample_text_clean)
print("Prediction:", prediction[0])

Prediction: negative


In [73]:
sample_text = ["The Product is not bad not good but average!"]
sample_text_clean = [clean_text(t) for t in sample_text]

prediction = loaded_model.predict(sample_text_clean)
print("Prediction:", prediction[0])

Prediction: positive


In [74]:
!pip -q install gradio scikit-learn pandas numpy joblib


In [75]:
import gradio as gr
import joblib

model = joblib.load("text_classifier.pkl")

def predict_label(text):
    cleaned = clean_text(text)
    pred = model.predict([cleaned])[0]
    return pred

app = gr.Interface(
    fn=predict_label,
    inputs=gr.Textbox(lines=5, placeholder="Type text here..."),
    outputs="text",
    title="🧠 NLP Text Classification",
    description="TF-IDF + Logistic Regression classifier"
)

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://adc32148f9d786f3df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
